In [1]:
#***************** Importation de données de Reddit *******************

import praw
import datetime
from RedditDocument import RedditDocument

docs = [] # contient les objets reddit

def fetch_reddit_posts(client_id, client_secret, user_agent, subreddit_name, nbposts=10):
    reddit = praw.Reddit(   # Connection to Reddit API
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent
    )

    subreddit = reddit.subreddit(subreddit_name) # specification du sujet

    for post in subreddit.hot(limit=nbposts): # the hottest nbposts
        readable_date = datetime.datetime.fromtimestamp( # conversion date
            post.created_utc
        ).strftime('%Y-%m-%d %H:%M:%S')

        doc = RedditDocument( # instanciation classe RedditDocument 
            titre=post.title,
            auteur=str(post.author),
            date=readable_date,
            url=post.url,
            texte=post.selftext,
            nb_comments=post.num_comments
        )

        docs.append(doc)  # save the added doc

    return docs



In [2]:
#***************** Importation de données de Arxiv *******************

import urllib.request as libreq
import xmltodict
from ArxivDocument import ArxivDocument

def fetch_arxiv(theme="machine+learning", max_results=10):
    url = f"http://export.arxiv.org/api/query?search_query=all:{theme}&start=0&max_results={max_results}"

    # Required USER AGENT
    req = libreq.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Python Script'}
    )

    data = libreq.urlopen(req).read()
    feed = xmltodict.parse(data)

    entries = feed["feed"].get("entry", [])

    # If single entry, wrap into list
    if not isinstance(entries, list):
        entries = [entries]

    docsA = []

    for entry in entries:
        title = entry["title"].strip()

        authors = (
            [a["name"] for a in entry["author"]]
            if isinstance(entry["author"], list)
            else [entry["author"]["name"]]
        )

        date = entry["published"]
        text = entry["summary"]

        url = None
        links = entry.get("link", [])
        if isinstance(links, dict):
            links = [links]
        for l in links:
            if l["@rel"] == "alternate":  # main HTML page
                url = l["@href"]

        doc = ArxivDocument(title, authors, date, url, text) # instanciation classe ArxivDocument
        docsA.append(doc)

    return docsA


In [3]:
from Corpus import Corpus
from RedditDocument import RedditDocument
from ArxivDocument import ArxivDocument
# import fetch_reddit_posts
# import fetch_arxiv

In [4]:
# #***************** Création du corpus ( données Reddit + Arxiv ) *******************

# def main():
#     corpus = Corpus("MySearchEngine")

#     # 1) Get Reddit docs
#     reddit_docs = fetch_reddit_posts(
#         client_id="MYp9TRuDb4U_nHdmnuvL9Q",
#         client_secret="FH1vXNJyBg_wxrQPsOohQqM5wuR1XA",
#         user_agent="WebScraping",
#         subreddit_name="MachineLearning",
#         nbposts=5
#     )

#     for doc in reddit_docs:
#         corpus.add_document(doc)

#     # 2) Get Arxiv docs
#     arxiv_docs = fetch_arxiv("MachineLearning", max_results=5)
#     for doc in arxiv_docs:
#         corpus.add_document(doc)

#     # 3) Save to CSV
#     corpus.save("corpus.csv")

#     print("DONE:", corpus)

# if __name__ == "__main__":
#     main()

# doc = Corpus.factory(texte, titre, auteur, date, "reddit")



In [5]:
# main.py
from Corpus import Corpus


def main():
    corpus = Corpus("Search_Engine")

    # --- REDDIT ---
    reddit_data = fetch_reddit_posts(
        client_id="MYp9TRuDb4U_nHdmnuvL9Q",
        client_secret="FH1vXNJyBg_wxrQPsOohQqM5wuR1XA",
        user_agent="WebScraping",
        subreddit_name="MachineLearning",
        nbposts=5
    )

    for d in reddit_data:
        doc = Corpus.factory(
            texte=d.texte,
            titre=d.titre,
            auteur=d.auteur,
            date=d.date,
            source="reddit",
            url=d.url,
            nb_comments=d.getNbComments()
        )
        corpus.add_document(doc)


    # --- ARXIV ---
    arxiv_data = fetch_arxiv("MachineLearning", max_results=5)

    for d in arxiv_data:
        doc = Corpus.factory(
            texte=d.texte,
            titre=d.titre,
            auteur=d.auteur,
            date=d.date,
            source="arxiv",
            url=d.url
        )
        corpus.add_document(doc)

    # SAVE
    corpus.save("corpus.csv")

    print("DONE:", corpus)


if __name__ == "__main__":
    main()


Corpus saved to corpus.csv (10 documents).
DONE: <Corpus 'Search_Engine' — 10 docs / 9 auteurs>
